# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này cài đặt hoàn chỉnh toàn bộ pipeline từ chuẩn bị dữ liệu, zero-shot E0, data collator mask loss, LoRA setup cho E1-E3, vòng huấn luyện, suy luận, ablation E4 và reward function E5.


## 0. Môi trường (Colab T4)
Chạy trên Google Colab với GPU T4 (16GB VRAM). Cài đặt các thư viện từ requirements-colab.txt.


In [8]:
!git clone https://github.com/khangnguyenthe18/K4-Track-4-Day10-NguyenTheKhang-2A202602964-VisionLanguageModel.git repo
%cd repo


Cloning into 'repo'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 72 (delta 17), reused 15 (delta 12), pack-reused 41 (from 1)
Receiving objects: 100% (72/72), 93.15 KiB | 18.63 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/repo


In [9]:
# Runtime -> Change runtime type -> T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt
!python -m pytest -q


GPU 0: Tesla T4 (UUID: GPU-b4cfaf95-3dce-16f9-f721-8f3efb1358e5)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.4 MB/s eta 0:00:00
...............F...........s.........................................    [100%]
=================================== FAILURES ===================================
__________ test_report_template_has_student_markers_and_ten_questions __________

    def test_report_template_has_student_markers_and_ten_questions():
>       assert "<điền" in DOCS["REPORT.md"]
E       AssertionError: assert '<điền' in 'Họ tên: Nguyễn Thế Khang - 2A202602964\n`dataset_version`: d10-21e4070301\n\n## Bảng tổng hợp (số của bạn, lấy từ `re... riêng các trọng số float32 của projector trong E2/E3 là bài học kỹ thuật quan trọng khi làm v

In [10]:
# Kaggle: dung Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHONG viet khoa vao notebook.
import os
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print('Da nap Kaggle credentials tu Colab Secrets.')
except Exception:
    print('Moi truong ngoai Colab Secrets.')


Moi truong ngoai Colab Secrets.


## 1. Dữ liệu: tải và chia split
Tai bo du lieu road damage va chia split co dinh theo phien quay (session) de chong ro ri du lieu giua cac frame lien ke.


In [11]:
!python scripts/prepare_data.py --kaggle


Dataset URL: https://www.kaggle.com/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes
License(s): MIT
100% 185M/185M [00:10<00:00, 18.8MB/s]

dataset_version : d10-21e4070301
  train  1422 ảnh  boxes={'pothole': 1074, 'crack': 1750, 'manhole': 691}
  val     276 ảnh  boxes={'pothole': 66, 'crack': 470, 'manhole': 103}
  test    311 ảnh  boxes={'pothole': 121, 'crack': 299, 'manhole': 163}


In [12]:
# Kiem tra cac split va xac nhan khong co phien nao bi trung lap giua cac split
from scripts.labcommon import read_jsonl

train_entries = read_jsonl('data/splits/train.jsonl')
val_entries = read_jsonl('data/splits/val.jsonl')
test_entries = read_jsonl('data/splits/test.jsonl')
print(f'Train samples: {len(train_entries)}, Val: {len(val_entries)}, Test: {len(test_entries)}')

train_sessions = {e['session'] for e in train_entries}
val_sessions = {e['session'] for e in val_entries}
test_sessions = {e['session'] for e in test_entries}
assert len(train_sessions & val_sessions) == 0, 'Ro ri giua train va val!'
assert len(train_sessions & test_sessions) == 0, 'Ro ri giua train va test!'
print('Kiem tra thanh cong: Khong co session nao xuat hien o 2 split!')


Train samples: 1422, Val: 276, Test: 311
Kiem tra thanh cong: Khong co session nao xuat hien o 2 split!


## 2. Nhãn: YOLO → JSON thang 0–1000
Quy doi toa do YOLO [xc, yc, w, h] chuan hoa [0,1] sang bounding box thang nguyen 0-1000 chuan Qwen3-VL.


In [13]:
import json

CLASSES = ['pothole', 'crack', 'manhole']
PROMPT = (
    "Locate road damages: pothole, crack, and manhole. "
    "Output a JSON list of objects with 'label' and 'bbox_2d' on a 0-1000 scale [x1, y1, x2, y2]."
)

def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo
    x1 = int(round(max(0.0, min(1.0, xc - w / 2.0)) * 1000))
    y1 = int(round(max(0.0, min(1.0, yc - h / 2.0)) * 1000))
    x2 = int(round(max(0.0, min(1.0, xc + w / 2.0)) * 1000))
    y2 = int(round(max(0.0, min(1.0, yc + h / 2.0)) * 1000))
    if x2 <= x1: x2 = min(1000, x1 + 1)
    if y2 <= y1: y2 = min(1000, y1 + 1)
    return {'label': label, 'bbox_2d': [x1, y1, x2, y2]}

def objects_to_answer(objects):
    boxes = [yolo_to_box(obj['label'], obj['yolo']) for obj in objects]
    boxes = sorted(boxes, key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))
    return json.dumps(boxes, separators=(',', ':'))

# Test chuyen doi
sample_obj = [{'label': 'pothole', 'yolo': [0.5, 0.5, 0.2, 0.2]}]
print('Sample ground-truth answer:', objects_to_answer(sample_obj))


Sample ground-truth answer: [{"label":"pothole","bbox_2d":[400,400,600,600]}]


## 3. E0: zero-shot
Nap mo hinh Qwen3-VL-4B-Instruct voi cau hinh luong tu 4-bit (bo qua visual va lm_head) va suy luan tren tap test.


In [17]:
import os
import torch
from PIL import Image
from transformers import BitsAndBytesConfig, Qwen3VLForConditionalGeneration, AutoProcessor
from scripts.labcommon import read_jsonl, write_jsonl

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=['visual', 'lm_head']
)

model_id = 'Qwen/Qwen3-VL-4B-Instruct'
try:
    processor = AutoProcessor.from_pretrained(model_id)
except Exception:
    from transformers import AutoTokenizer, AutoImageProcessor, Qwen3VLVideoProcessor, Qwen3VLProcessor
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    image_processor = AutoImageProcessor.from_pretrained(model_id)
    video_processor = Qwen3VLVideoProcessor()
    processor = Qwen3VLProcessor(
        image_processor=image_processor,
        tokenizer=tokenizer,
        video_processor=video_processor,
        chat_template=tokenizer.chat_template
    )
processor.chat_template = processor.tokenizer.chat_template
processor.tokenizer.padding_side = 'left'

model_e0 = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization,
    torch_dtype=torch.float16,
    device_map={'': 0}
)
model_e0.eval()

os.makedirs('work', exist_ok=True)
test_entries = read_jsonl('data/splits/test.jsonl')
e0_preds = []

with torch.no_grad():
    for entry in test_entries:
        img_path = os.path.join('data', entry['image_path'])
        image = Image.open(img_path).convert('RGB')
        messages = [{
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': PROMPT}
            ]
        }]
        inputs = processor.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt'
        ).to('cuda')
        prompt_len = inputs['input_ids'].shape[1]
        out = model_e0.generate(**inputs, max_new_tokens=256, do_sample=False)
        raw = processor.tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True)
        e0_preds.append({'image_id': entry['image_id'], 'raw': raw})

write_jsonl('work/e0.jsonl', e0_preds)
print('Đã lưu kết quả work/e0.jsonl.')


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Đã lưu kết quả work/e0.jsonl.


In [18]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json


E0 [test] n=311 valid=0.907 P=0.115 R=0.075 F1=0.091 CI95=0.068-0.119 meanIoU=0.613 FP/img=1.09
   pothole  P=0.538 R=0.116 F1=0.190
   crack    P=0.020 R=0.017 F1=0.018
   manhole  P=0.229 R=0.153 F1=0.184


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
Chi tinh loss tren phan tra loi (labels = -100 tren toan bo prompt va 220 visual tokens).


In [19]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class RoadDamageDataset(Dataset):
    def __init__(self, jsonl_path, data_dir='data', fraction=1.0):
        entries = read_jsonl(jsonl_path)
        if fraction < 1.0:
            entries = entries[:int(len(entries) * fraction)]
        self.entries = entries
        self.data_dir = data_dir

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        img_path = os.path.join(self.data_dir, entry['image_path'])
        image = Image.open(img_path).convert('RGB')
        answer = objects_to_answer(entry['objects'])
        return image, answer

def collate_fn(batch):
    batch_input_ids, batch_attention_mask, batch_labels = [], [], []
    pixel_values_list, image_grid_thw_list = [], []

    for image, answer in batch:
        user_msg = [{'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': PROMPT}]}]
        prompt_enc = processor.apply_chat_template(
            user_msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt'
        )
        p_len = prompt_enc['input_ids'].shape[1]

        full_msg = [
            {'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': PROMPT}]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': answer}]}
        ]
        full_enc = processor.apply_chat_template(
            full_msg, add_generation_prompt=False, tokenize=True, return_dict=True, return_tensors='pt'
        )

        input_ids = full_enc['input_ids'][0]
        attention_mask = full_enc['attention_mask'][0]
        labels = input_ids.clone()
        labels[:p_len] = -100

        batch_input_ids.append(input_ids)
        batch_attention_mask.append(attention_mask)
        batch_labels.append(labels)
        pixel_values_list.append(full_enc['pixel_values'])
        image_grid_thw_list.append(full_enc['image_grid_thw'])

    pad_id = processor.tokenizer.pad_token_id or 0
    input_ids_padded = pad_sequence(batch_input_ids, batch_first=True, padding_value=pad_id)
    attention_mask_padded = pad_sequence(batch_attention_mask, batch_first=True, padding_value=0)
    labels_padded = pad_sequence(batch_labels, batch_first=True, padding_value=-100)

    pixel_values = torch.cat(pixel_values_list, dim=0)
    image_grid_thw = torch.cat(image_grid_thw_list, dim=0)

    return {
        'input_ids': input_ids_padded,
        'attention_mask': attention_mask_padded,
        'labels': labels_padded,
        'pixel_values': pixel_values,
        'image_grid_thw': image_grid_thw
    }


## 5. Setup LoRA / đóng băng cho E1, E2, E3
Cau hinh target_modules cho LLM, mo full projector o float32 (E2/E3) va LoRA tren ViT (E3).


In [20]:
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup='E1', r=16, alpha=32, dropout=0.05):
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()

    llm_targets = r'model\.language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)'
    vit_targets = r'model\.visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2))'

    if setup in ['E1', 'E2']:
        targets = llm_targets
    elif setup == 'E3':
        targets = f'({llm_targets}|{vit_targets})'

    lora_cfg = LoraConfig(
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        target_modules=targets,
        task_type='CAUSAL_LM'
    )
    model = get_peft_model(model, lora_cfg)

    if setup in ['E2', 'E3']:
        for name, param in model.named_parameters():
            if 'visual.merger' in name or 'visual.deepstack_merger_list' in name:
                param.data = param.data.to(torch.float32)
                param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'[{setup}] Trainable parameters: {trainable:,}')
    return model, trainable


## 6. Vòng huấn luyện
Huan luyen 300 buoc toi uu voi AdamW, cosine schedule, GradScaler fp16, gradient checkpointing. Luu rieng Projector cho E2/E3.


In [21]:
import time
from torch.cuda.amp import GradScaler, autocast
from transformers import get_cosine_schedule_with_warmup

def train_experiment(setup_name, batch_size=2, grad_accum=4, max_steps=300, lr=2e-4, r=16, alpha=32, fraction=1.0):
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id, quantization_config=quantization, torch_dtype=torch.float16, device_map={'': 0}
    )
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model.config.use_cache = False

    model, trainable_params = apply_setup(model, setup=setup_name, r=r, alpha=alpha)

    train_ds = RoadDamageDataset('data/splits/train.jsonl', fraction=fraction)
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.0)
    scheduler = get_cosine_schedule_with_warmup(opt, num_warmup_steps=int(max_steps * 0.05), num_training_steps=max_steps)
    scaler = GradScaler('cuda')

    model.train()
    step = 0
    data_iter = iter(loader)

    while step < max_steps:
        opt.zero_grad()
        for _ in range(grad_accum):
            try:
                batch = next(data_iter)
            except StopIteration:
                data_iter = iter(loader)
                batch = next(data_iter)

            batch = {k: v.to('cuda') for k, v in batch.items()}
            with autocast(dtype=torch.float16):
                outputs = model(**batch)
                loss = outputs.loss / grad_accum

            if not torch.isfinite(loss):
                raise RuntimeError(f'Loss non-finite at step {step}: {loss.item()}')

            scaler.scale(loss).backward()

        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(opt)
        scaler.update()
        scheduler.step()
        step += 1

        if step % 50 == 0 or step == max_steps:
            print(f'Step {step}/{max_steps} | Loss: {loss.item() * grad_accum:.4f}')

    train_minutes = round((time.time() - start_time) / 60.0, 2)
    peak_vram_gb = round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2)

    save_dir = f'work/{setup_name.lower()}_adapter'
    model.save_pretrained(save_dir)

    # Luu rieng Projector neu la E2 hoac E3
    if setup_name in ['E2', 'E3']:
        proj_state = {
            k: v.cpu() for k, v in model.named_parameters()
            if ('visual.merger' in k or 'visual.deepstack_merger_list' in k) and v.requires_grad
        }
        torch.save(proj_state, f'work/{setup_name.lower()}_projector.pt')

    print(f'[{setup_name}] Train finished: {trainable_params:,} params, {peak_vram_gb} GB VRAM, {train_minutes} min.')
    return trainable_params, peak_vram_gb, train_minutes


## 7. Suy luận và chấm E1–E3
Nap lai adapter LoRA cung projector weights (neu co), suy luan tren test va cham bang scripts/score.py co metadata.


In [22]:
from peft import PeftModel

def infer_and_score(setup_name, trainable_params, peak_vram_gb, train_minutes):
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id, quantization_config=quantization, torch_dtype=torch.float16, device_map={'': 0}
    )
    model = PeftModel.from_pretrained(base_model, f'work/{setup_name.lower()}_adapter')

    if setup_name in ['E2', 'E3']:
        proj_path = f'work/{setup_name.lower()}_projector.pt'
        if os.path.exists(proj_path):
            proj_weights = torch.load(proj_path, map_location='cuda')
            state_dict = model.state_dict()
            for k, v in proj_weights.items():
                if k in state_dict:
                    state_dict[k].copy_(v)

    model.eval()
    test_entries = read_jsonl('data/splits/test.jsonl')
    preds = []

    with torch.no_grad():
        for entry in test_entries:
            img = Image.open(os.path.join('data', entry['image_path'])).convert('RGB')
            messages = [{'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': PROMPT}]}]
            inputs = processor.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt'
            ).to('cuda')
            p_len = inputs['input_ids'].shape[1]
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
            raw = processor.tokenizer.decode(out[0, p_len:], skip_special_tokens=True)
            preds.append({'image_id': entry['image_id'], 'raw': raw})

    pred_file = f'work/{setup_name.lower()}.jsonl'
    write_jsonl(pred_file, preds)

    out_json = f'report/evidence/{setup_name.lower()}.json'
    cmd = (
        f'python scripts/score.py --pred {pred_file} --name {setup_name} --out {out_json} '
        f'--meta trainable_params={trainable_params} '
        f'--meta peak_vram_gb={peak_vram_gb} '
        f'--meta train_minutes={train_minutes}'
    )
    print('Scoring:', cmd)
    os.system(cmd)

# Huan luyen va cham E1, E2, E3
# p1, v1, t1 = train_experiment('E1')
# infer_and_score('E1', p1, v1, t1)

# p2, v2, t2 = train_experiment('E2')
# infer_and_score('E2', p2, v2, t2)

# p3, v3, t3 = train_experiment('E3')
# infer_and_score('E3', p3, v3, t3)


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)
Khao sat bien rank tren E1: r=4 vs r=64. Ham reward IoU cho GRPO.


In [23]:
# Khao sat bien rank tren E1
# p4_4, v4_4, t4_4 = train_experiment('E1', r=4, alpha=8)
# infer_and_score('E4_rank_4', p4_4, v4_4, t4_4)

# p4_64, v4_64, t4_64 = train_experiment('E1', r=64, alpha=128)
# infer_and_score('E4_rank_64', p4_64, v4_64, t4_64)

# --- BONUS E5: Ham Reward IoU cho GRPO ---
from scripts.score import parse_output, iou_xyxy

def compute_iou_reward(completion, gt_boxes):
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if not gt_boxes and not pred:
        return 1.0
    if not gt_boxes or not pred:
        return 0.2

    pairs = []
    for i, g in enumerate(gt_boxes):
        for j, p in enumerate(pred):
            if g['label'] == p['label']:
                iou = iou_xyxy(g['bbox'], p['bbox'])
                if iou >= 0.5:
                    pairs.append((iou, i, j))

    pairs.sort(reverse=True)
    used_g, used_p = set(), set()
    total_iou = 0.0
    for iou, i, j in pairs:
        if i not in used_g and j not in used_p:
            used_g.add(i)
            used_p.add(j)
            total_iou += iou

    denom = max(len(gt_boxes), len(pred), 1)
    return 0.2 + 0.8 * (total_iou / denom)

# Unit test kiem chung reward
sample_gt = [{'label': 'pothole', 'bbox': (100, 100, 300, 300)}]
print('Reward hoan hao:', compute_iou_reward('[{"label":"pothole","bbox_2d":[100,100,300,300]}]', sample_gt))
print('Reward JSON hong:', compute_iou_reward('malformed', sample_gt))
print('Reward sai lop:', compute_iou_reward('[{"label":"crack","bbox_2d":[100,100,300,300]}]', sample_gt))


Reward hoan hao: 1.0
Reward JSON hong: 0.0
Reward sai lop: 0.2
